
# Árboles de decisión
### Material introductorio para alumnos de Ingeniería Mecatrónica

---

## Objetivo
Comprender de manera sencilla qué son los **árboles de decisión**, cómo funcionan y cómo pueden utilizarse en problemas básicos de clasificación dentro de Ingeniería Mecatrónica.

## ¿Qué aprenderás?
Al finalizar este notebook podrás:

- entender la lógica de un árbol de decisión,
- identificar sus partes principales,
- entrenar un modelo sencillo con Python,
- visualizar el árbol,
- interpretar sus reglas de decisión,
- aplicar esta técnica a problemas básicos de clasificación.

---

## Contexto en Mecatrónica
En Ingeniería Mecatrónica es común tomar decisiones con base en mediciones de sensores y variables de proceso, por ejemplo:

- **si la temperatura es alta y la vibración también, entonces hay riesgo de falla**,
- **si la corriente supera cierto valor, puede haber sobrecarga**,
- **si una pieza no cumple con ciertas medidas, se clasifica como defectuosa**,
- **si la presión es baja y el caudal también, puede existir una obstrucción**.

Un árbol de decisión se parece mucho a esa lógica: **divide los datos mediante preguntas simples** hasta llegar a una decisión final.



## 1. ¿Qué es un árbol de decisión?

Un árbol de decisión es un modelo de Machine Learning que organiza decisiones en forma de árbol.

Empieza con una pregunta en la parte superior y va separando los datos en ramas según ciertas condiciones.

Por ejemplo:

- ¿la vibración es mayor que 2.0?
- ¿la temperatura es mayor que 40 °C?
- ¿la corriente es mayor que 3 A?

Con una secuencia de preguntas, el modelo llega a una clasificación final.

### Idea general
Un árbol de decisión busca responder:

> **¿Qué regla sencilla permite separar mejor las clases?**



## 2. Partes de un árbol de decisión

Un árbol tiene elementos principales:

- **Nodo raíz**: primera pregunta del árbol.
- **Nodos internos**: preguntas intermedias.
- **Ramas**: caminos según la respuesta.
- **Hojas**: resultado final o clase predicha.

### Ejemplo conceptual
Un sistema podría seguir esta lógica:

1. ¿Temperatura > 40?
2. Si sí: ¿Vibración > 2.5?
3. Si ambas son altas: **Falla**
4. En otro caso: **Normal**

Eso es justamente el tipo de estructura que aprende un árbol de decisión.



## 3. ¿Para qué sirve?

Los árboles de decisión pueden usarse para:

- **clasificación**: decidir entre categorías,
- **regresión**: predecir valores numéricos.

En este notebook nos enfocaremos en **clasificación**.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

pd.set_option("display.precision", 3)



## 4. Ejemplo sencillo: detección de condición de falla en un motor

Supongamos que medimos dos variables en un motor:

- **Temperatura (°C)**
- **Vibración (mm/s)**

Y queremos clasificar el estado del sistema como:

- **0 = normal**
- **1 = posible falla**

Los siguientes datos son ficticios, pero razonables para fines didácticos.


In [ ]:

temperatura = np.array([28, 30, 32, 34, 35, 36, 38, 39, 40, 42, 44, 46, 48, 50], dtype=float)
vibracion  = np.array([0.8, 1.0, 1.1, 1.3, 1.5, 1.6, 1.9, 2.0, 2.2, 2.6, 2.9, 3.1, 3.5, 3.8], dtype=float)
falla      = np.array([0,   0,   0,   0,   0,   0,   0,   0,   1,   1,   1,   1,   1,   1], dtype=int)

df = pd.DataFrame({
    "Temperatura_C": temperatura,
    "Vibracion_mm_s": vibracion,
    "Falla": falla
})

df



## 5. Visualización inicial de los datos

Primero observamos cómo se distribuyen los datos.


In [ ]:

plt.figure(figsize=(8,5))
plt.scatter(df["Temperatura_C"], df["Vibracion_mm_s"], c=df["Falla"])
plt.title("Temperatura y vibración del motor")
plt.xlabel("Temperatura (°C)")
plt.ylabel("Vibración (mm/s)")
plt.grid(True)
plt.show()



En esta gráfica cada punto representa una observación del motor.

La idea del árbol será encontrar reglas que permitan separar los casos **normales** de los casos con **posible falla**.



## 6. Preparación de variables

En Machine Learning usamos:

- **X** para las entradas,
- **y** para la salida.


In [ ]:

X = df[["Temperatura_C", "Vibracion_mm_s"]]
y = df["Falla"]

X.head()



## 7. Entrenamiento del árbol de decisión

Usaremos `DecisionTreeClassifier` de `scikit-learn`.


In [ ]:

modelo = DecisionTreeClassifier(max_depth=3, random_state=42)
modelo.fit(X, y)

print("Modelo entrenado correctamente.")



### ¿Qué significa `max_depth=3`?
Indica la profundidad máxima del árbol.

- Un árbol muy pequeño puede ser demasiado simple.
- Un árbol muy grande puede memorizar demasiado los datos.

Por eso normalmente se controla su tamaño.



## 8. Predicciones con el modelo


In [ ]:

y_pred = modelo.predict(X)

resultado = df.copy()
resultado["Prediccion"] = y_pred
resultado



## 9. Evaluación básica


In [ ]:

acc = accuracy_score(y, y_pred)
matriz = confusion_matrix(y, y_pred)

print(f"Accuracy: {acc:.4f}")
print("\nMatriz de confusión:")
print(matriz)


In [ ]:

print(classification_report(y, y_pred, zero_division=0))



### Interpretación sencilla

- **Accuracy** indica qué proporción de casos clasificó correctamente el modelo.
- La **matriz de confusión** muestra aciertos y errores por clase.
- El **classification report** resume métricas útiles como precisión y recall.

En problemas reales, estas métricas ayudan a decidir si el modelo es confiable.



## 10. Visualización del árbol

Una de las mayores ventajas de los árboles de decisión es que pueden **verse** y **entenderse** fácilmente.


In [ ]:

plt.figure(figsize=(14,8))
plot_tree(
    modelo,
    feature_names=["Temperatura_C", "Vibracion_mm_s"],
    class_names=["Normal", "Falla"],
    filled=True,
    rounded=True
)
plt.title("Árbol de decisión para detección de falla")
plt.show()



## 11. Interpretación del árbol

Cada nodo del árbol contiene una condición, por ejemplo:

- `Temperatura_C <= 39.5`
- `Vibracion_mm_s <= 2.1`

Estas reglas separan gradualmente los datos.

### Lectura básica
Se puede leer de arriba hacia abajo:

- si se cumple la condición, se sigue una rama,
- si no se cumple, se sigue la otra,
- al llegar a una hoja, el modelo toma una decisión final.

Esto hace que el árbol sea muy intuitivo para aplicaciones de ingeniería.



## 12. Predicción para un nuevo caso

Supongamos un motor con:

- temperatura = **43 °C**
- vibración = **2.7 mm/s**

Queremos saber si el árbol lo clasifica como normal o como falla.


In [ ]:

nuevo_caso = pd.DataFrame({
    "Temperatura_C": [43],
    "Vibracion_mm_s": [2.7]
})

pred_nuevo = modelo.predict(nuevo_caso)[0]
prob_nuevo = modelo.predict_proba(nuevo_caso)[0]

print("Predicción:", pred_nuevo)
print("Probabilidades [Normal, Falla]:", prob_nuevo)



Si el resultado es:

- **0** → el modelo considera que el sistema está en condición normal,
- **1** → el modelo considera que existe posible falla.

También es útil observar las probabilidades para conocer qué tan seguro está el modelo.



## 13. Otro ejemplo breve: clasificación de piezas

Supongamos ahora una inspección de piezas basada en dos variables:

- **ancho**
- **peso**

Y queremos decidir si una pieza está:

- **0 = aceptada**
- **1 = defectuosa**


In [ ]:

ancho = np.array([9.8, 10.0, 10.1, 10.2, 10.3, 10.5, 10.7, 10.8, 11.0, 11.2], dtype=float)
peso  = np.array([48, 49, 50, 50.5, 51, 52.5, 54, 55, 57, 58], dtype=float)
defecto = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1], dtype=int)

df2 = pd.DataFrame({
    "Ancho_mm": ancho,
    "Peso_g": peso,
    "Defecto": defecto
})

X2 = df2[["Ancho_mm", "Peso_g"]]
y2 = df2["Defecto"]

modelo2 = DecisionTreeClassifier(max_depth=2, random_state=42)
modelo2.fit(X2, y2)

plt.figure(figsize=(12,6))
plot_tree(
    modelo2,
    feature_names=["Ancho_mm", "Peso_g"],
    class_names=["Aceptada", "Defectuosa"],
    filled=True,
    rounded=True
)
plt.title("Árbol de decisión para inspección de piezas")
plt.show()



## 14. Ventajas de los árboles de decisión

- Son fáciles de interpretar.
- Funcionan bien con reglas tipo “si-entonces”.
- No requieren escalado obligatorio de variables en muchos casos.
- Permiten visualizar el proceso de decisión.
- Son útiles como introducción al Machine Learning.

## 15. Limitaciones

- Pueden sobreajustarse si crecen demasiado.
- Un pequeño cambio en los datos puede cambiar la estructura del árbol.
- No siempre ofrecen el mejor desempeño frente a métodos más avanzados.
- En algunos problemas complejos se necesitan varios árboles o modelos más robustos.



## 16. Parámetros importantes

Algunos parámetros comunes son:

- **max_depth**: profundidad máxima del árbol,
- **min_samples_split**: mínimo de muestras para dividir un nodo,
- **min_samples_leaf**: mínimo de muestras en una hoja,
- **criterion**: criterio para decidir la mejor división.

En cursos introductorios, `max_depth` suele ser suficiente para empezar.



## 17. Conclusión

Los árboles de decisión son una técnica muy útil para iniciar en Machine Learning porque permiten construir modelos basados en reglas claras y fáciles de interpretar.

En Ingeniería Mecatrónica pueden aplicarse a:

- detección de fallas,
- clasificación de estados,
- control basado en condiciones,
- inspección de calidad,
- mantenimiento predictivo básico.

Además, ayudan a desarrollar intuición sobre cómo una máquina toma decisiones a partir de datos.



# Ejercicios propuestos

## Ejercicio 1. Comprensión conceptual
Responde con tus propias palabras:

1. ¿Qué es un árbol de decisión?
2. ¿Qué función tienen los nodos y las hojas?
3. ¿Por qué un árbol de decisión puede ser útil en Ingeniería Mecatrónica?

---

## Ejercicio 2. Sistema térmico
Crea un conjunto de datos con:

- temperatura,
- corriente del sistema,
- clase `0 = normal`, `1 = alarma`.

Después:

1. crea el DataFrame,
2. entrena un árbol de decisión,
3. visualiza el árbol,
4. interpreta sus reglas principales.

---

## Ejercicio 3. Clasificación de piezas
Simula datos de inspección con dos variables:

- largo de la pieza,
- peso de la pieza,

y clasifica como:

- `0 = aceptada`
- `1 = defectuosa`

Realiza:

1. entrenamiento del modelo,
2. predicción de nuevos casos,
3. evaluación básica,
4. explicación técnica del resultado.

---

## Ejercicio 4. Cambio de profundidad
Usa el mismo conjunto de datos y entrena tres modelos con:

- `max_depth=1`
- `max_depth=2`
- `max_depth=4`

Compara:

1. la forma del árbol,
2. el accuracy,
3. cuál parece más interpretable.

---

## Ejercicio 5. Reflexión aplicada
Menciona un caso real de Ingeniería Mecatrónica donde usarías un árbol de decisión y explica:

- qué variables medirías,
- cuál sería la salida,
- qué decisiones podría tomar el sistema.



# Actividad integradora sugerida

## Mini proyecto
Plantea un problema simple de clasificación relacionado con mecatrónica, por ejemplo:

- detección de falla en un motor,
- clasificación de piezas,
- estado normal o peligroso en un sistema térmico,
- identificación de sobrecarga en un actuador.

Entrega:

1. tabla de datos,
2. descripción del problema,
3. entrenamiento del árbol,
4. visualización del árbol,
5. predicción de al menos dos casos nuevos,
6. interpretación de las reglas,
7. conclusión técnica breve.
